<a href="https://colab.research.google.com/github/rkahrya1311/amazon-ml-challenge-2026/blob/kishor%2Fmodel/kishor_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# ML MODEL FOR ENTITY MATCHING
# ============================================================

import pandas as pd
import numpy as np
import joblib

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)


# ============================================================
# 1. UPLOAD ML FEATURE DATASET
# ============================================================

print("=" * 60)
print("UPLOAD ML FEATURE DATASET")
print("=" * 60)

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("\nDataset filename:", file_name)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


# ============================================================
# 2. SET TARGET AND FEATURES
# ============================================================

TARGET_COLUMN = "is_match"

FEATURE_COLUMNS = [
    "name_levenshtein",
    "name_token_similarity",
    "address_levenshtein",
    "address_token_similarity",
    "name_length_diff",
    "address_length_diff",
    "country_match"
]


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ============================================================
# 4. SEPARATE FEATURES AND TARGET
# ============================================================

X = df[FEATURE_COLUMNS]

y = df[TARGET_COLUMN]


print("\n" + "=" * 60)
print("FEATURE INFORMATION")
print("=" * 60)

print("Features used:")
for column in FEATURE_COLUMNS:
    print("-", column)

print("\nTarget column:", TARGET_COLUMN)


# ============================================================
# 5. CHECK LABEL DISTRIBUTION
# ============================================================

positive_samples = int((y == 1).sum())
negative_samples = int((y == 0).sum())

print("\n" + "=" * 60)
print("LABEL DISTRIBUTION")
print("=" * 60)

print("Positive samples (1):", positive_samples)
print("Negative samples (0):", negative_samples)
print("Total samples:", len(y))


# ============================================================
# 6. TRAIN / VALIDATION SPLIT
# ============================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("\n" + "=" * 60)
print("VALIDATION SPLIT")
print("=" * 60)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))

print("\nTraining labels:")
print(y_train.value_counts().sort_index())

print("\nValidation labels:")
print(y_valid.value_counts().sort_index())

print("\nSplit method:")
print("80% Training / 20% Validation")
print("Random row-level split")
print("Stratified by is_match")
print("NOT split by S1 entity_id")


# ============================================================
# 7. CREATE ML PIPELINE
# ============================================================

model = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )
])


# ============================================================
# 8. TRAIN MODEL
# ============================================================

print("\n" + "=" * 60)
print("MODEL TRAINING")
print("=" * 60)

model.fit(X_train, y_train)

print("Model training completed!")


# ============================================================
# 9. PREDICT PROBABILITY
# ============================================================

probabilities = model.predict_proba(
    X_valid
)[:, 1]


# ============================================================
# 10. TEST DIFFERENT THRESHOLDS
# ============================================================

threshold_results = []

best_threshold = 0.50
best_f05 = -1

for threshold in np.arange(0.10, 1.00, 0.01):

    predictions = (
        probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_valid,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_valid,
        predictions,
        zero_division=0
    )

    f05 = fbeta_score(
        y_valid,
        predictions,
        beta=0.5,
        zero_division=0
    )

    threshold_results.append({

        "threshold": round(float(threshold), 2),

        "precision": precision,

        "recall": recall,

        "f05": f05
    })

    if f05 > best_f05:

        best_f05 = f05

        best_threshold = float(threshold)


threshold_results_df = pd.DataFrame(
    threshold_results
)


# ============================================================
# 11. PRINT THRESHOLD RESULTS
# ============================================================

print("\n" + "=" * 60)
print("THRESHOLD RESULTS")
print("=" * 60)

print(
    threshold_results_df.to_string(
        index=False
    )
)


# ============================================================
# 12. FINAL VALIDATION PREDICTION
# ============================================================

final_predictions = (
    probabilities >= best_threshold
).astype(int)


# ============================================================
# 13. CALCULATE FINAL METRICS
# ============================================================

precision = precision_score(
    y_valid,
    final_predictions,
    zero_division=0
)

recall = recall_score(
    y_valid,
    final_predictions,
    zero_division=0
)

f05 = fbeta_score(
    y_valid,
    final_predictions,
    beta=0.5,
    zero_division=0
)

f1 = f1_score(
    y_valid,
    final_predictions,
    zero_division=0
)

cm = confusion_matrix(
    y_valid,
    final_predictions
)


# ============================================================
# 14. FINAL VALIDATION RESULTS
# ============================================================

print("\n" + "=" * 60)
print("FINAL VALIDATION RESULTS")
print("=" * 60)

print(
    "Best Threshold :",
    round(best_threshold, 2)
)

print(
    "Precision      :",
    round(precision, 4)
)

print(
    "Recall         :",
    round(recall, 4)
)

print(
    "F0.5           :",
    round(f05, 4)
)

print(
    "F1             :",
    round(f1, 4)
)

print("\nConfusion Matrix:")

print(cm)


# ============================================================
# 15. SAVE MODEL
# ============================================================

MODEL_FILE = (
    "entity_matching_logistic_regression.pkl"
)


model_package = {

    "model": model,

    "threshold": best_threshold,

    "feature_columns": FEATURE_COLUMNS,

    "target_column": TARGET_COLUMN
}


joblib.dump(
    model_package,
    MODEL_FILE
)


print("\n" + "=" * 60)
print("MODEL SAVED")
print("=" * 60)

print("Model file:", MODEL_FILE)


# ============================================================
# 16. INFERENCE FUNCTION
# ============================================================

def predict_matches(candidate_features):

    package = joblib.load(
        MODEL_FILE
    )

    trained_model = package["model"]

    threshold = package["threshold"]

    feature_columns = package[
        "feature_columns"
    ]


    # Check columns

    missing_columns = [

        column

        for column in feature_columns

        if column not in candidate_features.columns

    ]


    if missing_columns:

        raise ValueError(
            f"Missing feature columns: {missing_columns}"
        )


    # Select only ML features

    X_test = candidate_features[
        feature_columns
    ]


    # Predict probability

    probabilities = trained_model.predict_proba(
        X_test
    )[:, 1]


    # Apply threshold

    predictions = (
        probabilities >= threshold
    ).astype(int)


    # Create result

    result = candidate_features.copy()

    result["match_probability"] = probabilities

    result["prediction"] = predictions


    return result


# ============================================================
# 17. ML READY
# ============================================================

print("\n" + "=" * 60)
print("ML MODEL READY FOR INFERENCE")
print("=" * 60)

print(
    "Use:"
)

print(
    "result = predict_matches(candidate_features)"
)

print(
    "\nPrediction:"
)

print(
    "1 = MATCH"
)

print(
    "0 = NO MATCH"
)

UPLOAD ML FEATURE DATASET


Saving feature_engineered_100k_V2_labelled.csv to feature_engineered_100k_V2_labelled (1).csv

Dataset filename: feature_engineered_100k_V2_labelled (1).csv
Dataset shape: (100000, 12)
Columns: ['name_levenshtein', 'name_token_similarity', 'address_levenshtein', 'address_token_similarity', 'name_length_diff', 'address_length_diff', 'country_match', 's1_id', 'matched_id', 'matched_source', 'blocking_rule', 'is_match']

FEATURE INFORMATION
Features used:
- name_levenshtein
- name_token_similarity
- address_levenshtein
- address_token_similarity
- name_length_diff
- address_length_diff
- country_match

Target column: is_match

LABEL DISTRIBUTION
Positive samples (1): 254
Negative samples (0): 99746
Total samples: 100000

VALIDATION SPLIT
Training rows: 80000
Validation rows: 20000

Training labels:
is_match
0    79797
1      203
Name: count, dtype: int64

Validation labels:
is_match
0    19949
1       51
Name: count, dtype: int64

Split method:
80% Training / 20% Validation
Random row-lev